[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/implementation/notebooks/week01-solutions.ipynb)

> **Tip:** click the badge to open this notebook interactively in Google Colab and explore the cells live.

# Week 1 — Solutions & Live Exploration (Trainer Only)

Instructor-only companion to `docs/trainer/quiz-answers.md` and `docs/trainer/lab-solutions.md`.
**Do not distribute to learners** — it contains quiz answers and lab solutions.

Every code cell calls the repo's own `radar.*` functions — nothing is reimplemented — so the
answers are *computed*, not quoted, and you can perturb parameters mid-lesson to explore.

Run top-to-bottom. All randomness is seeded via `RadarConfig(seed=42)`.

## Setup

In [ ]:
import os
import subprocess
import sys

# Colab runs on a fresh VM each session, so clone the repo and install the
# package before `from radar import ...` works. Only runs in Colab (guarded
# so it never clones into a local checkout). Idempotent on re-run; update
# the branch/URL here if they change.
in_colab = "google.colab" in str(get_ipython().__class__)
if in_colab and not os.path.isdir("active-radar-tracker-basics"):
    subprocess.run(
        ["git", "clone", "-b", "implementation",
         "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"],
        check=True,
    )
if in_colab:
    os.chdir("active-radar-tracker-basics")
    subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
    # Ensure the cloned repository src/ is on sys.path so imports work even
    # if the editable install did not register on this fresh kernel.
    repo_src = os.path.abspath("src")
    if repo_src not in sys.path:
        sys.path.insert(0, repo_src)  # prioritize local modules

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from radar import signal_gen
from radar.config import RadarConfig

cfg = RadarConfig(seed=42)
print(cfg)

## Quiz 1 — Duty cycle

> With `τ = 20 us` and `T = 1 ms`, what is the duty cycle?

A radar can't listen while it shouts, so it alternates bursts with quiet gaps. Duty cycle is the
fraction of time the transmitter is on. Computed from the config, not from memory:

In [ ]:
duty = cfg.pulse_width_s / cfg.pri_s
print(f"duty cycle = {duty:.1%}")
print(f"quiet (listen) window = {(1 - duty):.1%} of the time")

## Quiz 2 — Round-trip delay (samples)

> A target at `R = 1000 m` — what round-trip delay (in samples at `fs = 20 MHz`)?

The wave travels out *and* back, so the delay is `2R/c`, then converted to samples by multiplying
by `fs`. This is the single most load-bearing number in Week 1 — every later stage indexes arrays by it.

In [ ]:
c = 3e8
t_delay = 2 * cfg.target_range_m / c
n_delay = round(t_delay * cfg.fs_hz)
print(f"round-trip delay = {t_delay * 1e6:.2f} us")
print(f"delay in samples = {n_delay}   (expected 133)")

## Quiz 3 — Why the factor of 2?

> Why does `R = c·τ/2` have a factor of 2?

Out and back. The measured delay is the *round trip*; range is half of it. Trivial, but the factor
of 2 is the most common unit bug in radar code — check it every time you convert delay ↔ range.

In [ ]:
R = c * t_delay / 2
assert np.isclose(R, cfg.target_range_m)
print(f"R = c·t_delay/2 = {R:.0f} m  (consistent with the configured target)")

## Quiz 4 — Range resolution

> What is the range resolution of a 5 MHz LFM chirp? Why is that better than a 20 us rectangular pulse?

After pulse compression, **bandwidth** sets resolution, not pulse width: `ΔR = c/(2B)`. The chirp
collects long-pulse energy (`τ·B = 100`) yet resolves like a short pulse.

In [ ]:
delta_r_lfm = c / (2 * cfg.bandwidth_hz)
delta_r_rect = c * cfg.pulse_width_s / 2
print(f"LFM  (B=5 MHz,  τ=20 us): ΔR = {delta_r_lfm:.0f} m")
print(f"Rect (τ=20 us):           ΔR = {delta_r_rect:.0f} m")
print(
    f"improvement = {delta_r_rect / delta_r_lfm:.0f}x  (time-bandwidth product τ·B = {cfg.bandwidth_hz * cfg.pulse_width_s:.0f})"
)

## Stage 1 recap — The transmit waveform

Generate the pulse and confirm its two defining properties: length = `τ·fs` samples, and the
instantaneous frequency sweeps `0 → B` across the pulse.

In [ ]:
pulse = signal_gen.lfm_chirp(cfg)
print(
    f"pulse samples = {pulse.size}  ({pulse.size / cfg.fs_hz * 1e6:.1f} us at {cfg.fs_hz / 1e6:.0f} MHz)"
)

t_us = np.arange(pulse.size) / cfg.fs_hz * 1e6
fig, ax = plt.subplots()
ax.plot(t_us, np.real(pulse))
ax.set_xlabel("time (us)")
ax.set_ylabel("Re{s(t)}")
ax.set_title("LFM chirp (real part)")
plt.show()

In [ ]:
phase = np.unwrap(np.angle(pulse))
f_inst = np.diff(phase) / (2 * np.pi) * cfg.fs_hz

fig, ax = plt.subplots()
ax.plot(t_us[1:], f_inst / 1e6)
ax.axhline(cfg.bandwidth_hz / 1e6, color="r", ls="--", lw=1, label="B")
ax.set_xlabel("time (us)")
ax.set_ylabel("instantaneous frequency (MHz)")
ax.set_title(
    f"sweep {f_inst[0] / 1e3:.0f} kHz → {f_inst[-1] / 1e6:.2f} MHz (B = {cfg.bandwidth_hz / 1e6:.0f} MHz)"
)
ax.legend()
plt.show()

## Stage 1 recap — The pulse train

64 pulses per CPI, each PRI 20,000 samples long, the pulse in the first 400 samples and silence
after — that silence is where the echo will land (Stage 2).

In [ ]:
train = signal_gen.pulse_train(cfg)
print(f"train shape = {train.shape}  (n_pulses, samples_per_pri)")
print(
    f"energy per pulse row: {np.sum(np.abs(train) > 0, axis=1).min()} samples (should be 400)"
)

## Stretch (Stage 1) — Pulse-compression preview

Correlate the chirp with itself (its autocorrelation — the same operation a matched filter
performs in Stage 3, and nothing from Stage 3 is needed here). The 20 us, 400-sample chirp
collapses into a spike whose half-width is ~1/B (0.2 us, 4 samples). Measure the compressed
peak, the half-width, and the compression ratio (pulse samples ÷ half-width ≈ τ·B = 100).

In [ ]:
from scipy.signal import correlate

pulse = signal_gen.lfm_chirp(cfg)
# autocorrelation = matched filtering (scipy's correlate conjugates its kernel,
# so the transmit chirp itself is the matched filter's impulse response)
comp = np.abs(correlate(pulse, pulse, mode="same"))
peak = comp.max()
peak_idx = comp.argmax()

# compressed main-lobe half-width = distance from peak to the first null
# (null ~= value below 2% of the peak; expected at 1/B = 4 samples)
threshold = 0.02 * peak
first_null = np.argmax(comp[peak_idx:] < threshold)

t_us = np.arange(pulse.size) / cfg.fs_hz * 1e6
fig, ax = plt.subplots()
ax.plot(t_us, comp / peak)
ax.axvline(t_us[peak_idx + first_null], color="r", ls="--", lw=1, label="first null")
ax.set_xlabel("time (us)")
ax.set_ylabel("|compressed| (normalized)")
ax.set_title("Pulse compression: the 20 us chirp collapses to a ~1/B spike")
ax.legend()
plt.show()

print(f"compressed peak = {peak:.0f}  (= N = {pulse.size}, coherent sum of pulse samples)")
print(f"main-lobe half-width (peak -> first null) = {first_null} samples = {first_null/cfg.fs_hz*1e6:.2f} us")
print(f"expect 1/B = {cfg.fs_hz/cfg.bandwidth_hz:.0f} samples = {1/cfg.bandwidth_hz*1e6:.2f} us")
print(f"compression ratio = pulse/half-width = {pulse.size/first_null:.0f}  (expect tau*B = {cfg.bandwidth_hz*cfg.pulse_width_s:.0f})")

## Stage 2 — Moving target simulation

The channel turns the transmit pulse into a delayed, attenuated, noisy echo. The echo of
R = 1000 m lands at sample `round(2R/c·fs) = 133`. Against unit-power noise, the echo
amplitude is `10^(snr_db/20)` (10 at 20 dB). Note: `velocity_mps` is carried now but
consumed in Stage 4 (Doppler) — delay/attenuation/noise are all this stage does.

In [ ]:
from radar import channel

target = channel.Target(range_m=1000.0, velocity_mps=40.0, snr_db=20.0)
rng = np.random.default_rng(cfg.seed)
rx = channel.simulate_channel(signal_gen.transmit_waveform(cfg), [target], cfg, rng)

n_delay = round(2 * target.range_m / 3e8 * cfg.fs_hz)
print(f"expected echo onset = {n_delay} samples (round(2R/c*fs), 133 for 1000 m)")
print(f"rx shape = {rx.shape}  (n_pulses, samples_per_pri)")

t_us = np.arange(rx.shape[1]) / cfg.fs_hz * 1e6
fig, ax = plt.subplots()
ax.plot(t_us, np.abs(rx[0]))
ax.axvline(n_delay / cfg.fs_hz * 1e6, color="r", ls="--", lw=1, label="expected echo (sample 133)")
ax.set_xlim(0, 40)
ax.set_xlabel("time (us)")
ax.set_ylabel("|rx|")
ax.set_title("Received signal: echo at 133 samples buried in noise")
ax.legend()
plt.show()

noise_power = np.mean(np.abs(rx[0, 2000:]) ** 2)
signal_power = np.mean(np.abs(rx[0, n_delay : n_delay + 400]) ** 2) - noise_power
snr_measured = 10 * np.log10(signal_power / noise_power)
print(f"noise power = {noise_power:.2f}  (expect 1)")
print(f"signal power = {signal_power:.1f}  (expect 100 at 20 dB)")
print(f"measured SNR = {snr_measured:.1f} dB  (expect {target.snr_db:.0f} dB)")

## Stretch (Stage 2) — Two targets in `simulate_channel`

The channel sums one echo per target into the same PRI, then adds a single noise draw. With
targets spaced > 3 km the 400-sample echoes don't overlap, so each delay is visible: 1000 m ->
sample 133, 5000 m -> sample 667. (Targets closer than one pulse length, 3000 m, can't be
separated by eye — that's exactly what the Stage 3 matched filter resolves.)


In [ ]:
targets = [
    channel.Target(range_m=1000.0, velocity_mps=0.0, snr_db=20.0),
    channel.Target(range_m=5000.0, velocity_mps=0.0, snr_db=20.0),
]
rx2 = channel.simulate_channel(signal_gen.transmit_waveform(cfg), targets, cfg, rng)

onsets = []
for t in targets:
    n = round(2 * t.range_m / 3e8 * cfg.fs_hz)
    onsets.append(n)
    print(f"target at {t.range_m:.0f} m -> echo at sample {n}")

fig, ax = plt.subplots()
t_us = np.arange(rx2.shape[1]) / cfg.fs_hz * 1e6
ax.plot(t_us, np.abs(rx2[0]))
for n in onsets:
    ax.axvline(n / cfg.fs_hz * 1e6, color="r", ls="--", lw=1)
ax.set_xlim(0, 60)
ax.set_xlabel("time (us)")
ax.set_ylabel("|rx|")
ax.set_title("Two targets: echoes at both round-trip delays")
plt.show()


## Stage 3 — Matched filter & range estimation

Correlate the received echo with the transmit pulse: the 400-sample pulse compresses into a
spike whose position is the range. The peak lands at `delay + len(pulse)//2 = 133 + 200` and
the detected range (997.5 m) is within one 7.5 m range bin of the 1000 m truth. Processing
gain is N = 400 (26 dB), so a 20 dB echo becomes a ~46 dB detection.

In [ ]:
from radar import channel, receiver

pulse = signal_gen.lfm_chirp(cfg)
target = channel.Target(range_m=1000.0, velocity_mps=40.0, snr_db=20.0)
rng = np.random.default_rng(cfg.seed)
rx = channel.simulate_channel(signal_gen.transmit_waveform(cfg), [target], cfg, rng)
mf = receiver.matched_filter(rx, pulse)

peak = int(np.argmax(np.abs(mf[0])))
n_pulse = round(cfg.pulse_width_s * cfg.fs_hz)
print(f"compressed peak at sample {peak} (expect delay 133 + offset 200 = 333)")
print(f"measured range = {receiver.range_from_delay(peak - n_pulse // 2, cfg):.1f} m")

for d in receiver.detect_peaks(mf, cfg):
    print(f"detection: range = {d.range_m:.1f} m, snr = {d.snr_db:.1f} dB")

mag = np.abs(mf[0])
p = np.argmax(mag)
half = mag[p] / 2
l, r = p, p
while l > 0 and mag[l] > half:
    l -= 1
while r < len(mag) - 1 and mag[r] > half:
    r += 1
print(f"half-power width = {r - l} samples (pulse was {n_pulse} samples)")

fig, ax = plt.subplots()
t_us = np.arange(len(mf[0])) / cfg.fs_hz * 1e6
ax.plot(t_us, np.abs(mf[0]), label="|matched|")
ax.axvline(133 / cfg.fs_hz * 1e6 + n_pulse / 2 / cfg.fs_hz * 1e6, color="r", ls="--", lw=1)
ax.set_xlim(0, 40)
ax.set_xlabel("time (us)")
ax.set_ylabel("|matched|")
ax.set_title("Matched filter: compressed spike at the target delay")
ax.legend()
plt.show()

## Stretch (Stage 3) — Two targets through `detect_peaks`

Feed the two-target echo (Stage 2 stretch) through the full Stage 3 pipeline: matched filter
then `detect_peaks`. The compressed peaks land at delay + 200 (133 + 200 and 667 + 200), and
each detection reports its own range and SNR — both within one 7.5 m range bin of truth.


In [ ]:
rx_st = channel.simulate_channel(signal_gen.transmit_waveform(cfg), targets, cfg, rng)
mf_st = receiver.matched_filter(rx_st, pulse)

for d in sorted(receiver.detect_peaks(mf_st, cfg), key=lambda d: d.range_m):
    print(f"detection: range = {d.range_m:.1f} m, snr = {d.snr_db:.1f} dB")

fig, ax = plt.subplots()
r_m = (np.arange(len(mf_st[0])) - n_pulse // 2) * 3e8 / (2 * cfg.fs_hz)
ax.plot(r_m / 1e3, np.abs(mf_st[0]))
for t in targets:
    ax.axvline(t.range_m / 1e3, color="r", ls="--", lw=1)
ax.set_xlim(0, 6)
ax.set_xlabel("range (km)")
ax.set_ylabel("|matched|")
ax.set_title("Stretch: two compressed peaks at their true ranges")
plt.show()


## Lab L1 — SNR sweep

Sweep input SNR 30 → -24 dB, run the full pipeline at each level, and watch **range accuracy**.
Two things to notice: (1) detection survives far below the "eyeball" level because the matched
filter adds 26 dB of processing gain — the target only drops out once SNR_in + 26 < 10 dB, i.e.
around -16 dB; (2) within the visible range the error stays within one 7.5 m range bin, until
noise corrupts the peak. False alarms (detections far from the target) creep in as SNR falls —
the CFAR preview: quality depends on the threshold policy, not just SNR.

In [ ]:
snrs = np.arange(30, -25, -3.0)
rows = []
for snr_db in snrs:
    rng = np.random.default_rng(cfg.seed + int(snr_db))
    tgt = channel.Target(range_m=1000.0, velocity_mps=40.0, snr_db=float(snr_db))
    rx = channel.simulate_channel(signal_gen.transmit_waveform(cfg), [tgt], cfg, rng)
    mf = receiver.matched_filter(rx, pulse)
    dets = receiver.detect_peaks(mf, cfg, threshold_db=10.0)
    near = [d for d in dets if abs(d.range_m - 1000.0) < 50.0]
    err = min(abs(d.range_m - 1000.0) for d in near) if near else None
    rows.append((snr_db, err, len(dets) - len(near)))

print(f"{"SNR_in(dB)":>11} {"range_err(m)":>13} {"false":>6}")
for snr_db, err, false in rows:
    err_s = f"{err:.1f}" if err is not None else "missed"
    print(f"{snr_db:>11.1f} {err_s:>13} {false:>6d}")

fig, ax = plt.subplots()
ax.plot(snrs, [e if e is not None else np.nan for _, e, _ in rows], "o-")
ax.axhline(7.5, color="r", ls="--", lw=1, label="one range bin (7.5 m)")
ax.set_xlabel("input SNR (dB)")
ax.set_ylabel("|measured - true| range error (m)")
ax.set_title("Lab L1: range accuracy vs input SNR (fixed 10 dB threshold)")
ax.legend()
plt.show()

## Free play

Change **one** parameter, re-run the relevant cell, and explain the effect. Seeded, so results are
reproducible:

- `cfg.pulse_width_s` — what happens to the number of samples per pulse?
- `cfg.pulse_type = "rect"` — how does the waveform (and its spectrum) differ from the chirp?
- `cfg.target_range_m` — where does the echo peak land once stages 2–3 are in place?
- `cfg.fs_hz` — what breaks if you drop the sample rate below `2·B`?